# Cuaderno 05: Simulación de Orquestación de Inferencia, Visualización e Interfaz (Fases 7 y 8)

**Responsable:** Integrante 3 (Agente LLM Local e Interfaz Gráfica en Streamlit)

## 1. Propósito Académico y Alcance
Este cuaderno documenta y valida de extremo a extremo el flujo completo de inferencia en tiempo real implementado en `app/servicios/orquestador_inferencia.py`, los componentes visuales en Plotly (`app/componentes/reproductor_onda.py` y `app/componentes/graficos_plotly.py`) y el módulo de diagnóstico en lenguaje natural (`src/agente/`).

La cadena ejecutada sigue estrictamente el contrato arquitectónico del proyecto:
1. **Validación e ingesta en memoria:** Verificación de integridad con `soundfile` (`es_archivo_valido`).
2. **Procesamiento Digital de Señales (DSP):** Estandarización a mono ($22\,050\text{ Hz}$) y extracción del vector de 28 descriptores acústicos.
3. **Inferencia predictiva y explicabilidad:** Escalado (`StandardScaler`), predicción (`XGBoost`), inversión logarítmica (`expm1`) y atribución local `SHAP` (o activación segura de *modo demostración* si los artefactos `.joblib` aún no están en `modelos_guardados/`).
4. **Diagnóstico determinista y agente LLM:** Detección de desviaciones frente a umbrales virales (`detector_deficiencias.py`) y redacción de recomendaciones técnicas mediante Ollama local (`llama3.2:1b`) o motor de respaldo basado en reglas (`formateador_respuesta.py`).

In [ ]:
import io
import sys
from pathlib import Path

import numpy as np
import soundfile as sf

# Garantizar que la raiz del repositorio este en sys.path al ejecutar desde notebooks/
RAIZ_PROYECTO = Path.cwd().resolve()
if not (RAIZ_PROYECTO / "config" / "settings.py").exists():
    RAIZ_PROYECTO = RAIZ_PROYECTO.parent
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

from config import settings
from src.agente.cliente_ollama import ClienteOllama
from app.servicios.orquestador_inferencia import MotorInferencia
from app.componentes.reproductor_onda import leer_metadatos_audio, _grafico_onda
from app.componentes.graficos_plotly import (
    grafico_radar,
    grafico_velocimetro,
    grafico_barras_shap,
)

cliente_probe = ClienteOllama()
print(f"Raiz del proyecto       : {RAIZ_PROYECTO}")
print(f"Tasa de muestreo (Hz)   : {settings.SAMPLE_RATE}")
print(f"Modelo Ollama config.   : {settings.OLLAMA_MODEL} ({settings.OLLAMA_HOST})")
print(f"Ollama en linea         : {cliente_probe.esta_disponible()}")

## 2. Generación de Audio Sintético de Prueba en Memoria
Para garantizar **100 % de reproducibilidad** sin depender de archivos multimedia externos ni descargas de red, sintetizamos en memoria una señal acústica controlada:
- **Frecuencia de muestreo:** $f_s = 22\,050\text{ Hz}$ (`settings.SAMPLE_RATE`).
- **Duración total:** $6.0\text{ s}$ (superior al mínimo de $2.0\text{ s}$ exigido por `validador_medios.py`).
- **Silencio inicial intencional:** $0.80\text{ s}$ de amplitud cero al inicio. Dado que supera el umbral crítico `settings.UMBRAL_SILENCIO_CRITICO_SEC = 0.35 s`, permite comprobar que tanto el extractor DSP como el detector de deficiencias identifiquen esta anomalía de retención.
- **Contenido rítmico y armónico:** Portadora senoidal de $220\text{ Hz}$ con su segundo armónico ($440\text{ Hz}$) modulada por una envolvente periódica a $100\text{ BPM}$.

In [ ]:
sr = settings.SAMPLE_RATE
duracion_seg = 6.0
silencio_objetivo_seg = 0.80
bpm_objetivo = 100.0

t = np.linspace(0.0, duracion_seg, int(sr * duracion_seg), endpoint=False)
envolvente_ritmica = 0.5 * (1.0 + np.sin(2.0 * np.pi * (bpm_objetivo / 60.0) * t))
senal = (
    0.65 * np.sin(2.0 * np.pi * 220.0 * t)
    + 0.25 * np.sin(2.0 * np.pi * 440.0 * t)
) * envolvente_ritmica

# Insertar silencio inicial controlado de 0.80 s
muestras_silencio = int(silencio_objetivo_seg * sr)
senal[:muestras_silencio] = 0.0
senal = (0.85 * senal / np.max(np.abs(senal))).astype(np.float32)

# Serializar a WAV en memoria (sin crear archivos permanentes en disco)
buffer_wav = io.BytesIO()
sf.write(buffer_wav, senal, sr, format="WAV")
audio_bytes = buffer_wav.getvalue()

print(f"Audio sintetico generado en memoria: {len(audio_bytes) / 1024:.1f} KB")
print(f"Muestras totales: {len(senal)} | Silencio inicial inyectado: {silencio_objetivo_seg:.2f} s")

## 3. Ejecución Extremo a Extremo del Orquestador (`MotorInferencia`)
Invocamos directamente la clase `MotorInferencia` de `app/servicios/orquestador_inferencia.py` mediante su método `analizar_bytes()`. Internamente, el orquestador:
1. Crea un directorio temporal (`tempfile.TemporaryDirectory`) que se elimina automáticamente al concluir.
2. Valida la cabecera WAV y extrae las 28 características acústicas con `librosa`.
3. Ejecuta la predicción con XGBoost y la atribución SHAP (o activa `modo_demo = True` si aún no se han entrenado los artefactos en `modelos_guardados/`).
4. Evalúa las desviaciones y consulta al agente generativo o al motor de respaldo.

In [ ]:
motor = MotorInferencia()
resultado = motor.analizar_bytes(audio_bytes, "audio_sintetico_prueba.wav")

assert resultado.exito, f"Error en inferencia: {resultado.mensaje_error}"

print("=== RESUMEN DE EJECUCION DEL ORQUESTADOR ===")
print(f"Exito                   : {resultado.exito}")
print(f"Modo demostracion       : {resultado.modo_demo}")
print(f"Umbrales oficiales      : {motor.umbrales_oficiales}")
if resultado.engagement_estimado is not None:
    print(f"Engagement estimado     : {resultado.engagement_estimado * 100:.2f} %")
else:
    print("Engagement estimado     : N/A (pendiente artefactos de Persona 2)")

print("\nTiempos de ejecucion (s):")
for etapa, seg in resultado.tiempos.items():
    print(f"  - {etapa:10s}: {seg:.4f} s")

print("\nMetricas acusticas clave extraidas (de 28 totales):")
claves_resumen = [
    "tempo_bpm",
    "initial_silence_sec",
    "silence_ratio",
    "rms_mean",
    "dynamic_range",
    "spectral_centroid",
    "beat_regularity",
]
for k in claves_resumen:
    print(f"  - {k:22s}: {resultado.vector[k]:.4f}")

## 4. Visualización de la Señal Temporal y Gráfico de Radar en Plotly
Reutilizamos exactamente las funciones de visualización de la interfaz Streamlit:
- **`leer_metadatos_audio()` y `_grafico_onda()`** (`app/componentes/reproductor_onda.py`): grafican la forma de onda temporal en el eje del tiempo, sobre la cual resaltamos en rojo la zona exacta de silencio inicial detectada (`initial_silence_sec`).
- **`grafico_radar()`** (`app/componentes/graficos_plotly.py`): compara los 5 ejes acústicos normalizados de la pista analizada frente a la mediana viral (`p50`). Cuando el modelo real y SHAP están activos, también se despliegan `grafico_velocimetro()` y `grafico_barras_shap()`.

In [ ]:
meta_audio = leer_metadatos_audio(audio_bytes)
fig_onda = _grafico_onda(meta_audio["onda_reducida"], meta_audio["paso_seg"])

# Resaltar el silencio inicial detectado por el modulo DSP
silencio_detectado = resultado.vector["initial_silence_sec"]
fig_onda.add_vrect(
    x0=0.0,
    x1=silencio_detectado,
    fillcolor="#ef4444",
    opacity=0.28,
    line_width=1,
    line_color="#ef4444",
    annotation_text=f"Silencio inicial ({silencio_detectado:.2f} s)",
    annotation_position="top left",
)
fig_onda.update_layout(
    title="Forma de onda temporal y deteccion de silencio inicial",
    paper_bgcolor="#0d0f14",
    plot_bgcolor="#151821",
    font=dict(color="#e5e7eb"),
    height=260,
)
fig_onda.show()

# Grafico de radar de 5 ejes usando la funcion real del repositorio
fig_radar = grafico_radar(resultado.vector, resultado.umbrales)
fig_radar.update_layout(paper_bgcolor="#0d0f14", font=dict(color="#e5e7eb"))
fig_radar.show()

# Si el modelo y SHAP estan disponibles, mostrar tambien velocimetro y barras SHAP
if resultado.engagement_estimado is not None:
    fig_gauge = grafico_velocimetro(resultado.engagement_estimado * 100)
    fig_gauge.update_layout(paper_bgcolor="#0d0f14", font=dict(color="#e5e7eb"))
    fig_gauge.show()

if resultado.explicacion_shap and resultado.explicacion_shap.get("contribuciones"):
    fig_shap = grafico_barras_shap(resultado.explicacion_shap["contribuciones"])
    fig_shap.update_layout(paper_bgcolor="#0d0f14", plot_bgcolor="#151821", font=dict(color="#e5e7eb"))
    fig_shap.show()

## 5. Diagnóstico de Deficiencias y Recomendaciones del Agente
El bloque determinista (`detector_deficiencias.py`) confronta las métricas extraídas contra los percentiles virales `[p25, p75]`. Con esa lista estructurada y anclada numéricamente:
- Si el servidor local de Ollama responde en el puerto `11434`, el modelo `llama3.2:1b` redacta 3 acciones concretas de edición sonora.
- Si Ollama no está encendido o no responde dentro del tiempo límite, `formateador_respuesta.py` activa automáticamente el **motor de respaldo basado en reglas**, garantizando continuidad operativa en la interfaz.

In [ ]:
print("=== DEFICIENCIAS DETECTADAS ===")
if not resultado.deficiencias:
    print("Sin desviaciones criticas ni moderadas fuera del rango viral.")
else:
    for idx, d in enumerate(resultado.deficiencias, start=1):
        shap_txt = "N/A" if d["impacto_shap"] is None else f"{d['impacto_shap']:+.4f}"
        print(
            f"{idx}. [{d['gravedad'].upper()}] {d['etiqueta']} ({d['variable']}): "
            f"valor={d['valor']} {d['unidad']} | rango viral=[{d['p25']}, {d['p75']}] "
            f"| direccion={d['direccion']} | SHAP={shap_txt}"
        )

diag = resultado.diagnostico
print("\n=== REPORTE DEL AGENTE DE DIAGNOSTICO ===")
print(f"Fuente utilizada        : {diag['fuente']}")
if diag["fuente"] == "ollama":
    print(f"Modelo / Duracion       : {diag.get('modelo')} ({diag.get('duracion_seg')} s)")
elif diag.get("motivo"):
    print(f"Motivo del respaldo     : {diag['motivo']}")

print("\nRecomendaciones de edicion sonora:")
for i, rec in enumerate(diag["recomendaciones"], start=1):
    print(f"  {i}. {rec}")

## 6. Conclusiones para Sustentación ante el Tribunal
1. **Integración desacoplada y sin archivos residuales:** `MotorInferencia` encapsula la ejecución completa en `analizar_bytes()` y `analizar_url()`, limpiando los directorios temporales al finalizar cada petición.
2. **Tolerancia a fallos y continuidad operativa:** El sistema opera de forma íntegra tanto con Ollama activo como en modo fuera de línea (activando el motor heurístico de `formateador_respuesta.py`), y soporta la transición transparente desde *modo demostración* hacia inferencia real en cuanto se depositan los binarios de la Persona 2 en `modelos_guardados/`.
3. **Trazabilidad entre señal física y lenguaje natural:** El silencio inicial inyectado ($0.80\text{ s}$) es detectado con precisión en el dominio temporal, visualizado en Plotly y traducido en una recomendación accionable prioritaria sin alucinaciones numéricas.